
# 14 — LangGraph Stateful Agents and Memory

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Related course notebooks:**
- `langchain_agents_and_tools.ipynb`
- `LCEL_conversation_history_and_memory.ipynb`
- `LCEL_conditional_logic_branching_and_merging.ipynb`
- `LCEL_runnable_interface.ipynb`

**Additional reference:**
[LangGraph Documentation](https://docs.langchain.com/oss/python/langgraph/overview)

## Goal

Build a small stateful customer-support workflow using
LangGraph.

The notebook covers:

1. Defining shared workflow state.
2. Creating graph nodes.
3. Connecting nodes using edges.
4. Routing using conditional edges.
5. Validating results and applying a bounded correction.
6. Defining tools and tool schemas.
7. Executing and inspecting graph state.
8. Streaming workflow updates.
9. Saving state using a checkpointer.
10. Keeping different conversation threads separate.
11. Understanding the difference between a graph
    and a model-driven agent.

The main workflow runs offline using deterministic
Python functions.

This is intentional: graph execution, state updates,
routing and checkpointing can be understood without
requiring a language model.

The final section explains how a model-driven agent
can use the same architecture.



## 1. LangChain vs. LangGraph

LangChain provides reusable components such as:

- Models.
- Prompts.
- Tools.
- Retrievers.
- Runnable workflows.

LangGraph provides explicit control over a workflow's
steps, shared state and transitions.

Key terminology:

**Node:** A function that performs one step.

**Edge:** A connection between nodes.

**State:** Shared information that nodes can read
and update.

**START:** The entry point of a graph.

**END:** The point where execution finishes.

**Conditional edge:** Chooses the next node
based on the current state.

**Checkpointer:** Saves snapshots of graph state.

**Thread ID:** Identifies a particular execution
history or conversation.

A graph is not automatically an AI agent.

Our first graph uses fixed Python rules.
Later, an LLM could make decisions inside nodes
or select tools dynamically.


In [1]:

%pip install -q "langgraph>=1,<2" "langchain-core>=1,<2" "pydantic>=2,<3"



## 2. Define the Workflow State

We will process customer-support requests.

Each request moves through these stages:

Input → Classification → Policy lookup → Draft answer

Then the answer is validated.

If it is valid, the workflow finishes.

If it is invalid, a correction step runs.

A state dictionary stores information such as:

- The original customer message.
- The selected category.
- The relevant policy.
- The drafted response.
- Whether validation succeeded.
- The number of corrections.
- The execution log.

The graph nodes return updates to this shared state.

This avoids manually passing many independent
variables between every step.


In [2]:

from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import (
    StateGraph,
    START,
    END
)
from langgraph.checkpoint.memory import InMemorySaver


class SupportState(TypedDict, total=False):
    message: str
    category: str
    policy: str
    answer: str
    valid: bool
    corrections: int
    log: Annotated[list[str], add]


POLICIES = {
    "returns": (
        "Unused items may be returned within 30 days."
    ),
    "delivery": (
        "Standard shipping usually takes "
        "3–5 business days."
    ),
    "damage": (
        "Customers can contact support "
        "about damaged items."
    ),
    "general": (
        "Contact support for questions not "
        "covered by the available policies."
    )
}

print("State and policy definitions ready.")


State and policy definitions ready.


In [3]:

def classify_request(state: SupportState):
    text = state["message"].lower()

    if "return" in text or "refund" in text:
        category = "returns"

    elif "damaged" in text or "broken" in text:
        category = "damage"

    elif "delivery" in text or "late" in text:
        category = "delivery"

    else:
        category = "general"

    return {
        "category": category,
        "log": [f"Classified as {category}"]
    }


def retrieve_policy(state: SupportState):
    policy = POLICIES[state["category"]]

    return {
        "policy": policy,
        "log": ["Retrieved relevant policy"]
    }


def draft_answer(state: SupportState):
    # This is a deliberately basic offline draft.
    # It is not generated by an LLM.
    return {
        "answer": state["policy"],
        "corrections": 0,
        "log": ["Created policy-based draft"]
    }


def validate_answer(state: SupportState):
    answer = state.get("answer", "").strip()

    # Our local rule requires an explicit reference
    # to the user's issue as well as the policy.
    valid = (
        bool(answer)
        and answer.startswith("Regarding your request:")
    )

    return {
        "valid": valid,
        "log": [f"Validation result: {valid}"]
    }


def correct_answer(state: SupportState):
    corrected = (
        "Regarding your request: "
        + state["policy"]
    )

    return {
        "answer": corrected,
        "corrections": state.get("corrections", 0) + 1,
        "log": ["Corrected response format"]
    }


print("Five graph nodes defined.")


Five graph nodes defined.



## 3. Connect the Graph

Our workflow will have five nodes:

1. `classify`
2. `retrieve`
3. `draft`
4. `validate`
5. `correct`

After validation, a conditional edge decides:

- Valid → END
- Invalid and correction allowed → correct
- Invalid after correction limit → END

The correction node returns to validation.

This creates a bounded loop.

Without a stopping rule, a correction workflow
could repeatedly call the same steps indefinitely.

The graph explicitly controls which operations
can execute next.


In [4]:

def choose_next_step(state: SupportState):
    if state.get("valid", False):
        return "finish"

    if state.get("corrections", 0) >= 1:
        return "finish"

    return "correct"


builder = StateGraph(SupportState)

builder.add_node("classify", classify_request)
builder.add_node("retrieve", retrieve_policy)
builder.add_node("draft", draft_answer)
builder.add_node("validate", validate_answer)
builder.add_node("correct", correct_answer)

builder.add_edge(START, "classify")
builder.add_edge("classify", "retrieve")
builder.add_edge("retrieve", "draft")
builder.add_edge("draft", "validate")

builder.add_conditional_edges(
    "validate",
    choose_next_step,
    {
        "finish": END,
        "correct": "correct"
    }
)

builder.add_edge("correct", "validate")

graph = builder.compile()

print("Graph compiled successfully.")


Graph compiled successfully.


In [5]:

result = graph.invoke({
    "message": "My item arrived damaged.",
    "log": []
})

print("Category:", result["category"])
print("Policy:", result["policy"])
print("Final answer:", result["answer"])
print("Valid:", result["valid"])
print("Corrections:", result["corrections"])

print("\nExecution log:")
for entry in result["log"]:
    print("-", entry)


assert result["category"] == "damage"
assert result["valid"] is True
assert result["corrections"] == 1
assert result["answer"].startswith(
    "Regarding your request:"
)

print("\nGraph execution tests passed.")


Category: damage
Policy: Customers can contact support about damaged items.
Final answer: Regarding your request: Customers can contact support about damaged items.
Valid: True
Corrections: 1

Execution log:
- Classified as damage
- Retrieved relevant policy
- Created policy-based draft
- Validation result: False
- Corrected response format
- Validation result: True

Graph execution tests passed.



## 4. Stream the Execution Steps

Streaming does not always mean streaming words
from a language model.

LangGraph can also stream execution updates.

For example, it can show when each node completes
and what that node changed.

This is useful for debugging workflows with
multiple branches or repeated steps.

We use `stream_mode="updates"` to inspect
the updates produced by each node.

No model requests are needed.


In [6]:

for update in graph.stream(
    {
        "message": "My delivery is late.",
        "log": []
    },
    stream_mode="updates"
):
    print(update)


{'classify': {'category': 'delivery', 'log': ['Classified as delivery']}}
{'retrieve': {'policy': 'Standard shipping usually takes 3–5 business days.', 'log': ['Retrieved relevant policy']}}
{'draft': {'answer': 'Standard shipping usually takes 3–5 business days.', 'corrections': 0, 'log': ['Created policy-based draft']}}
{'validate': {'valid': False, 'log': ['Validation result: False']}}
{'correct': {'answer': 'Regarding your request: Standard shipping usually takes 3–5 business days.', 'corrections': 1, 'log': ['Corrected response format']}}
{'validate': {'valid': True, 'log': ['Validation result: True']}}



## 5. Tools and Agent Actions

A tool is a function exposed through a schema
so that a model or agent can request its execution.

The course uses `@tool`, `StructuredTool` and
Pydantic schemas for tool arguments.

The distinction is:

**Tool definition:** Describe what a function does
and what arguments it accepts.

**Tool selection:** Decide which function should run.

**Tool execution:** Python actually runs the function.

A language model may select a tool, but merely
requesting a tool does not execute arbitrary Python.

The application or agent runtime must perform the
tool call and return the result.

Here we define and execute a tool directly.
This verifies the tool interface without
claiming that an LLM selected it.


In [7]:

from langchain_core.tools import tool


@tool
def lookup_policy(category: str) -> str:
    """Return the approved support policy for a category."""
    return POLICIES.get(
        category,
        POLICIES["general"]
    )


print("Tool name:", lookup_policy.name)
print("Tool description:", lookup_policy.description)

print("\nArgument schema:")
print(lookup_policy.args_schema.model_json_schema())

result = lookup_policy.invoke({
    "category": "returns"
})

print("\nTool result:", result)

assert "30 days" in result

print("Tool invocation passed.")


Tool name: lookup_policy
Tool description: Return the approved support policy for a category.

Argument schema:
{'description': 'Return the approved support policy for a category.', 'properties': {'category': {'title': 'Category', 'type': 'string'}}, 'required': ['category'], 'title': 'lookup_policy', 'type': 'object'}

Tool result: Unused items may be returned within 30 days.
Tool invocation passed.



## 6. Conversation Memory and Checkpointing

A checkpointer stores snapshots of graph state.

This allows the application to retrieve state from
a previous invocation.

LangGraph uses a `thread_id` to distinguish
different conversation histories.

For example:

- Thread A remembers its own messages.
- Thread B has a separate history.

An `InMemorySaver` stores checkpoints in Python memory.

It is useful for demonstrations but does not survive
restarting the Colab runtime.

For durable persistence, an application would use
a persistent checkpointer backed by appropriate
storage.

This example demonstrates actual graph checkpointing,
not merely storing messages in a Python list.


In [8]:

from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage


def local_reply(state: MessagesState):
    latest = state["messages"][-1].content

    return {
        "messages": [
            AIMessage(
                content=f"Received your message: {latest}"
            )
        ]
    }


memory_builder = StateGraph(MessagesState)
memory_builder.add_node("reply", local_reply)

memory_builder.add_edge(START, "reply")
memory_builder.add_edge("reply", END)

checkpointer = InMemorySaver()

memory_graph = memory_builder.compile(
    checkpointer=checkpointer
)


thread_a = {
    "configurable": {
        "thread_id": "conversation-a"
    }
}

thread_b = {
    "configurable": {
        "thread_id": "conversation-b"
    }
}

memory_graph.invoke(
    {
        "messages": [
            HumanMessage(content="Hello from A")
        ]
    },
    config=thread_a
)

memory_graph.invoke(
    {
        "messages": [
            HumanMessage(content="Second message from A")
        ]
    },
    config=thread_a
)

memory_graph.invoke(
    {
        "messages": [
            HumanMessage(content="Hello from B")
        ]
    },
    config=thread_b
)


history_a = memory_graph.get_state(thread_a)
history_b = memory_graph.get_state(thread_b)

print("Thread A messages:")
for message in history_a.values["messages"]:
    print(type(message).__name__, ":", message.content)

print("\nThread B messages:")
for message in history_b.values["messages"]:
    print(type(message).__name__, ":", message.content)


assert len(history_a.values["messages"]) == 4
assert len(history_b.values["messages"]) == 2

print("\nCheckpoint isolation tests passed.")


Thread A messages:
HumanMessage : Hello from A
AIMessage : Received your message: Hello from A
HumanMessage : Second message from A
AIMessage : Received your message: Second message from A

Thread B messages:
HumanMessage : Hello from B
AIMessage : Received your message: Hello from B

Checkpoint isolation tests passed.



## 7. Follow-Up Questions and Conversational Retrieval

The conversation-history lesson in the course
introduces another important problem.

Consider:

User: "Where does James work?"
User: "How old is he?"

The second question is not self-contained.

A conversation-aware retrieval system can use
previous messages to rewrite it as:

"How old is James?"

Then it retrieves relevant documents using the
standalone question.

A typical workflow is:

Conversation history + follow-up question
→ Contextualize the question
→ Retrieve documents
→ Generate a grounded answer

This is different from simply remembering messages.

**Checkpointing** saves conversation state.

**Question contextualization** uses that state to
interpret the current request.

The course uses an LLM to rewrite follow-up
questions and demonstrates older
`ConversationBufferMemory` abstractions.

Our graph shows modern checkpointing. An LLM-powered
contextualization node would be an optional extension.

### Agent vs. Fixed Workflow

Our graph follows predetermined Python rules.

A model-driven agent can dynamically choose actions,
request tools, inspect their results and continue.

LangChain's `create_agent()` provides a higher-level
interface to this pattern and uses LangGraph internally.

A custom StateGraph provides more explicit control
over individual nodes and transitions.

Neither approach is universally better. The choice
depends on how much control the application requires.


In [9]:

test_messages = {
    "I want to return this": "returns",
    "The product is broken": "damage",
    "My delivery is late": "delivery",
    "Can you help me?": "general"
}

for message, expected_category in test_messages.items():
    output = graph.invoke({
        "message": message,
        "log": []
    })

    assert output["category"] == expected_category
    assert output["valid"] is True
    assert output["corrections"] <= 1

    print(
        f"{expected_category:8} | {output['answer']}"
    )


assert (
    len(memory_graph.get_state(thread_a).values["messages"])
    == 4
)

assert (
    len(memory_graph.get_state(thread_b).values["messages"])
    == 2
)

print("\nAll graph and checkpoint tests passed.")
print("Model API requests: 0")


returns  | Regarding your request: Unused items may be returned within 30 days.
damage   | Regarding your request: Customers can contact support about damaged items.
delivery | Regarding your request: Standard shipping usually takes 3–5 business days.
general  | Regarding your request: Contact support for questions not covered by the available policies.

All graph and checkpoint tests passed.
Model API requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Defining LangGraph state using `TypedDict`.
- Accumulating state updates with a reducer.
- Building workflow nodes from Python functions.
- Connecting nodes with ordinary and conditional edges.
- Creating a bounded validation/correction loop.
- Compiling and invoking a graph.
- Streaming node updates.
- Defining a structured LangChain tool.
- Executing tools locally.
- Creating checkpoints using `InMemorySaver`.
- Isolating conversation history with `thread_id`.
- Testing graph behavior with Python assertions.

### How This Extends Notebook 13

Notebook 13 focused on composing reusable operations
with LangChain and LCEL.

Notebook 14 focuses on explicit state, transitions,
loops and persistence using LangGraph.

LCEL is suitable for many linear and branching
pipelines.

LangGraph provides additional control for workflows
requiring cycles, state management or checkpoints.

### Important Limitations

- The main graph uses deterministic routing rather
  than model-generated decisions.
- The correction loop demonstrates control flow,
  not actual LLM self-correction.
- The tool is invoked directly by Python; it is
  not dynamically selected by an LLM.
- The local reply node does not reason about
  previous conversation messages.
- The in-memory checkpointer does not persist
  after the Python runtime terminates.
- The example does not implement a production
  retrieval system, human approval or durable storage.

### Conclusion

LangGraph represents a workflow as connected nodes
that share and update state.

Conditional transitions make decisions explicit,
while checkpoints allow execution history to be
retained between invocations.

This notebook demonstrates the core mechanics
of stateful workflows independently of model behavior.

A model-driven agent can build on these foundations
by using language models inside nodes for
classification, reasoning, tool selection and
response generation.
